# 03b — Klasyczne ML: eksperymenty 1–3

Grid reprezentacja × model, strategie multi-label, optymalizacja progów. Wymaga `TW_FEATURES.pkl` z 03a.

## Setup

In [ ]:
import os
import time
import json
import pickle
import warnings
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix, hstack as sparse_hstack, issparse

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.metrics import (
    f1_score, hamming_loss, jaccard_score, accuracy_score, precision_score, recall_score,
)
from sklearn.model_selection import KFold
import lightgbm as lgb

from thesis_lib import evaluate, cache_features, optimal_thresholds, log_runs_from_df

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300

print(f"sklearn, lightgbm, gensim — wszystkie klasyczne, bez transformerów.")

In [2]:
# --- Constants ---
EMOTIONS = ["radość", "smutek", "zaufanie", "wstręt", "strach", "gniew", "przeczuwanie", "zdziwienie"]
RANDOM_STATE = 42

PROCESSED_DIR = Path("../data/processed")
CACHE_DIR     = Path("../data/features")
RESULTS_DIR   = Path("../data/results")
FIGURES_DIR   = Path("../figures")
for d in [CACHE_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Cache:   {CACHE_DIR}")
print(f"Results: {RESULTS_DIR}")
print(f"Figures: {FIGURES_DIR}")

Cache:   ../data/features
Results: ../data/results
Figures: ../figures


In [3]:
# --- Load processed data ---
tw_train = pd.read_csv(PROCESSED_DIR / "twitteremo_train.csv").reset_index(drop=True)
tw_val   = pd.read_csv(PROCESSED_DIR / "twitteremo_val.csv").reset_index(drop=True)
tw_test  = pd.read_csv(PROCESSED_DIR / "twitteremo_test.csv").reset_index(drop=True)

go_train = pd.read_csv(PROCESSED_DIR / "go_emotions_train.csv").reset_index(drop=True)
go_val   = pd.read_csv(PROCESSED_DIR / "go_emotions_val.csv").reset_index(drop=True)
go_test  = pd.read_csv(PROCESSED_DIR / "go_emotions_test.csv").reset_index(drop=True)

for df in [tw_train, tw_val, tw_test, go_train, go_val, go_test]:
    df["clean_text"] = df["clean_text"].fillna("")

y_tw_train = tw_train[EMOTIONS].values
y_tw_val   = tw_val[EMOTIONS].values
y_tw_test  = tw_test[EMOTIONS].values
y_go_train = go_train[EMOTIONS].values
y_go_val   = go_val[EMOTIONS].values
y_go_test  = go_test[EMOTIONS].values

print(f"TwitterEmo:    train={len(tw_train):,}, val={len(tw_val):,}, test={len(tw_test):,}")
print(f"GoEmotions PL: train={len(go_train):,}, val={len(go_val):,}, test={len(go_test):,}")

TwitterEmo:    train=28,684, val=5,737, test=1,435
GoEmotions PL: train=34,276, val=6,855, test=1,714


In [6]:
# --- Load features built in 03a ---
with open(CACHE_DIR / "TW_FEATURES.pkl", "rb") as f:
    TW_FEATURES = pickle.load(f)
print("Loaded TW_FEATURES:", list(TW_FEATURES))

Loaded TW_FEATURES: ['tfidf_word', 'tfidf_char', 'tfidf_wordchar', 'fasttext', 'lsa', 'lda', 'stats', 'nrc', 'combined']


In [ ]:
# --- MLflow tracking (file-based, no server) ---
import mlflow

# Newer MLflow refuses the file-store backend unless explicitly opted in.
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")

MLFLOW_DIR = Path("../mlruns").resolve()
mlflow.set_tracking_uri(f"file:{MLFLOW_DIR}")
mlflow.set_experiment("classical_ml_thesis")

print(f"MLflow -> {MLFLOW_DIR}  (eksperyment: classical_ml_thesis)")

## 3. Zestaw modeli

10 klasyfikatorów: 3 liniowe, 3 ensemble, 1 gradient boosting, 1 sieć, 1 instance-based, 1 probabilistyczny.

In [8]:
def make_models(class_weight: str | None = "balanced") -> dict:
    return {
        "logreg": LogisticRegression(
            max_iter=1000, C=1.0, class_weight=class_weight,
            solver="liblinear", random_state=RANDOM_STATE,
        ),
        "linearsvc": LinearSVC(
            C=1.0, class_weight=class_weight, max_iter=2000, random_state=RANDOM_STATE,
        ),
        "ridge": RidgeClassifier(
            alpha=1.0, class_weight=class_weight, random_state=RANDOM_STATE,
        ),
        "complement_nb": ComplementNB(alpha=0.5),
        "random_forest": RandomForestClassifier(
            n_estimators=200, n_jobs=-1, class_weight=class_weight, random_state=RANDOM_STATE,
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=200, n_jobs=-1, class_weight=class_weight, random_state=RANDOM_STATE,
        ),
        "lightgbm": lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=63,
            n_jobs=-1, class_weight=class_weight, random_state=RANDOM_STATE, verbose=-1,
        ),
        "hist_gb": HistGradientBoostingClassifier(
            max_iter=200, learning_rate=0.05, random_state=RANDOM_STATE,
            class_weight=class_weight,
        ),
        "mlp": MLPClassifier(
            hidden_layer_sizes=(256, 128), max_iter=50, early_stopping=True,
            random_state=RANDOM_STATE,
        ),
        "knn": KNeighborsClassifier(n_neighbors=15, n_jobs=-1, metric="cosine"),
    }


print("Modele:", list(make_models().keys()))

Modele: ['logreg', 'linearsvc', 'ridge', 'complement_nb', 'random_forest', 'extra_trees', 'lightgbm', 'hist_gb', 'mlp', 'knn']


## 4. Eksperyment 1: reprezentacja × model

Pełny grid bez kombinacji niekompatybilnych:
- ComplementNB — tylko cechy nieujemne (TF-IDF, LDA)
- KNN/MLP/HistGB — nie skalują się na dużym sparse TF-IDF
- RF/ExtraTrees — wolne na dużym sparse
- HistGradientBoosting — nie obsługuje sparse

In [9]:
# Ujednolicone definicje kompatybilności
SPARSE_LIN = {"logreg", "linearsvc", "ridge", "complement_nb", "lightgbm"}
DENSE_FAST = {"logreg", "linearsvc", "ridge", "random_forest", "extra_trees", "lightgbm", "hist_gb", "mlp", "knn"}
LDA_SET    = DENSE_FAST | {"complement_nb"}  # LDA outputs are non-negative probabilities

COMPATIBLE = {
    "tfidf_word":      SPARSE_LIN,
    "tfidf_char":      SPARSE_LIN,
    "tfidf_wordchar":  {"logreg", "linearsvc", "ridge", "complement_nb"},  # lightgbm zbyt wolny na ~68k sparse
    "fasttext":        DENSE_FAST,
    "lsa":             DENSE_FAST,
    "lda":             LDA_SET,
    "stats":           DENSE_FAST,
    "nrc":             DENSE_FAST,
    "combined":        {"logreg", "linearsvc", "ridge", "lightgbm"},
}


def fit_predict_ovr(model, X_train, y_train, X_eval) -> np.ndarray:
    clf = OneVsRestClassifier(model, n_jobs=1)
    clf.fit(X_train, y_train)
    return clf.predict(X_eval)


def run_grid(features_dict: dict, y_train: np.ndarray, y_val: np.ndarray) -> pd.DataFrame:
    rows = []
    models = make_models()
    total = sum(len(v) for v in COMPATIBLE.values())
    done = 0
    for rep_name, feats in features_dict.items():
        for model_name in COMPATIBLE.get(rep_name, set()):
            done += 1
            print(f"[{done:2d}/{total}] {rep_name:14s} × {model_name:14s} ", end="", flush=True)
            t0 = time.time()
            try:
                y_pred = fit_predict_ovr(models[model_name], feats["train"], y_train, feats["val"])
                m = evaluate(y_val, y_pred)
                m.update({"representation": rep_name, "model": model_name,
                          "time_s": round(time.time() - t0, 1)})
                rows.append(m)
                print(f"F1-Macro={m['f1_macro']:.3f} ({m['time_s']}s)")
            except Exception as e:
                print(f"FAILED: {type(e).__name__}: {e}")
    return pd.DataFrame(rows)


tw_grid = run_grid(TW_FEATURES, y_tw_train, y_tw_val).sort_values("f1_macro", ascending=False).reset_index(drop=True)
tw_grid.to_csv(RESULTS_DIR / "exp1_grid_twitteremo.csv", index=False)
print("\n=== Top 15 ===")
display(tw_grid.head(15))

[ 1/64] tfidf_word     × lightgbm       

F1-Macro=0.344 (43.1s)
[ 2/64] tfidf_word     × ridge          

F1-Macro=0.359 (0.5s)
[ 3/64] tfidf_word     × complement_nb  

F1-Macro=0.328 (0.1s)
[ 4/64] tfidf_word     × linearsvc      

F1-Macro=0.354 (1.4s)
[ 5/64] tfidf_word     × logreg         

...

F1-Macro=0.378 (67.3s)
[62/64] combined       × ridge          

F1-Macro=0.382 (25.3s)
[63/64] combined       × logreg         

F1-Macro=0.392 (34.4s)
[64/64] combined       × lightgbm       

F1-Macro=0.352 (206.5s)

=== Top 15 ===


,f1_macro,f1_micro,f1_weighted,precision_macro,recall_macro,hamming_loss,jaccard_macro,subset_accuracy,representation,model,time_s
0,0.445084,0.542398,0.560269,0.373042,0.575749,0.145329,0.299329,0.334147,tfidf_char,logreg,9.3
1,0.439342,0.547246,0.556001,0.389066,0.515297,0.134151,0.295361,0.356807,tfidf_wordchar,logreg,11.1
2,0.430384,0.536750,0.546315,0.387700,0.502206,0.137463,0.286918,0.337633,tfidf_char,ridge,7.9
3,0.426384,0.541033,0.544577,0.407811,0.468245,0.130752,0.284702,0.351926,tfidf_char,linearsvc,12.2
4,0.418901,0.569518,0.561194,0.469008,0.413785,0.113605,0.285585,0.409970,tfidf_char,lightgbm,975.1
5,0.413069,0.527483,0.528793,0.399631,0.440417,0.130926,0.273326,0.343908,tfidf_wordchar,ridge,10.1
6,0.404985,0.525574,0.522681,0.416521,0.407381,0.126111,0.267991,0.356458,tfidf_wordchar,linearsvc,10.7
7,0.392020,0.499738,0.521228,0.317756,0.539897,0.166550,0.258692,0.286735,combined,logreg,34.4
8,0.382165,0.488971,0.499717,0.328153,0.468157,0.156484,0.248648,0.289873,combined,ridge,25.3
9,0.378049,0.485704,0.491069,0.337818,0.434972,0.150885,0.245267,0.301377,combined,linearsvc,67.3


In [ ]:
# Heatmapa F1-Macro + czas uczenia
import sys
sys.path.insert(0, str(Path("..") / "experiments"))
from gen_fig_exp1_grid import build_figure

fig = build_figure(tw_grid)
fig.savefig(FIGURES_DIR / "exp1_grid_heatmap.pdf", bbox_inches="tight")
plt.show()


In [11]:
# Best per representation, best per model — for thesis tables
best_per_rep = tw_grid.loc[tw_grid.groupby("representation")["f1_macro"].idxmax()].sort_values("f1_macro", ascending=False).reset_index(drop=True)
best_per_model = tw_grid.loc[tw_grid.groupby("model")["f1_macro"].idxmax()].sort_values("f1_macro", ascending=False).reset_index(drop=True)

print("=== Najlepszy model per reprezentacja ===")
display(best_per_rep[["representation", "model", "f1_macro", "f1_micro", "jaccard_macro", "time_s"]])

print("\n=== Najlepsza reprezentacja per model ===")
display(best_per_model[["model", "representation", "f1_macro", "f1_micro", "jaccard_macro", "time_s"]])

best_per_rep.to_csv(RESULTS_DIR / "exp1_best_per_representation.csv", index=False)
best_per_model.to_csv(RESULTS_DIR / "exp1_best_per_model.csv", index=False)

=== Najlepszy model per reprezentacja ===


,representation,model,f1_macro,f1_micro,jaccard_macro,time_s
0,tfidf_char,logreg,0.445084,0.542398,0.299329,9.3
1,tfidf_wordchar,logreg,0.439342,0.547246,0.295361,11.1
2,combined,logreg,0.392020,0.499738,0.258692,34.4
3,tfidf_word,logreg,0.373934,0.474397,0.242776,0.7
4,fasttext,hist_gb,0.334085,0.419565,0.215790,9.6
5,lsa,hist_gb,0.314181,0.402723,0.198584,15.2
6,stats,hist_gb,0.277902,0.331123,0.173343,1.4
7,nrc,hist_gb,0.241078,0.281561,0.144401,1.3
8,lda,hist_gb,0.229802,0.276089,0.137171,1.3



=== Najlepsza reprezentacja per model ===


,model,representation,f1_macro,f1_micro,jaccard_macro,time_s
0,logreg,tfidf_char,0.445084,0.542398,0.299329,9.3
1,ridge,tfidf_char,0.430384,0.536750,0.286918,7.9
2,linearsvc,tfidf_char,0.426384,0.541033,0.284702,12.2
3,lightgbm,tfidf_char,0.418901,0.569518,0.285585,975.1
4,complement_nb,tfidf_wordchar,0.355255,0.523888,0.235798,0.3
5,hist_gb,fasttext,0.334085,0.419565,0.215790,9.6
6,mlp,lsa,0.227129,0.360267,0.139673,38.2
7,knn,fasttext,0.186577,0.368517,0.115949,15.6
8,extra_trees,nrc,0.164917,0.210092,0.092814,5.0
9,random_forest,nrc,0.158307,0.204575,0.088648,5.8


## 5. Eksperyment 2: strategie multi-label

Na najlepszej parze z eksp. 1: Binary Relevance (`OneVsRestClassifier`), ClassifierChain w kolejności malejącej częstości etykiet, ClassifierChain losowy (kontrola).

In [12]:
best_row = tw_grid.iloc[0]
best_rep, best_model_name = best_row["representation"], best_row["model"]
print(f"Najlepszy z Eksp. 1: {best_rep} × {best_model_name} (F1-Macro={best_row['f1_macro']:.3f})")

X_train_b = TW_FEATURES[best_rep]["train"]
X_val_b   = TW_FEATURES[best_rep]["val"]


def make_base() -> Any:
    return make_models()[best_model_name]


freq_order = list(np.argsort(-y_tw_train.sum(axis=0)))

strategies = {
    "binary_relevance":         OneVsRestClassifier(make_base(), n_jobs=1),
    "classifier_chain_freq":    ClassifierChain(make_base(), order=freq_order, random_state=RANDOM_STATE),
    "classifier_chain_random":  ClassifierChain(make_base(), order="random", random_state=RANDOM_STATE),
}

ml_rows = []
for name, clf in strategies.items():
    print(f"  {name} ...", end=" ", flush=True)
    t0 = time.time()
    clf.fit(X_train_b, y_tw_train)
    y_pred = clf.predict(X_val_b)
    if hasattr(y_pred, "toarray"):
        y_pred = y_pred.toarray()
    y_pred = np.asarray(y_pred).astype(int)
    m = evaluate(y_tw_val, y_pred)
    m.update({"strategy": name, "time_s": round(time.time() - t0, 1)})
    ml_rows.append(m)
    print(f"F1-Macro={m['f1_macro']:.3f} ({m['time_s']}s)")

ml_df = pd.DataFrame(ml_rows)
ml_df.to_csv(RESULTS_DIR / "exp2_multilabel_strategies.csv", index=False)
display(ml_df[["strategy", "f1_macro", "f1_micro", "jaccard_macro", "subset_accuracy", "time_s"]])

Najlepszy z Eksp. 1: tfidf_char × logreg (F1-Macro=0.445)
  binary_relevance ... 

F1-Macro=0.445 (9.2s)
  classifier_chain_freq ... 

F1-Macro=0.413 (13.8s)
  classifier_chain_random ... 

F1-Macro=0.418 (14.4s)


,strategy,f1_macro,f1_micro,jaccard_macro,subset_accuracy,time_s
0,binary_relevance,0.445084,0.542398,0.299329,0.334147,9.2
1,classifier_chain_freq,0.412786,0.531156,0.276078,0.335890,13.8
2,classifier_chain_random,0.417813,0.514150,0.277326,0.339550,14.4


## 6. Eksperyment 3: progi per etykieta

Próg 0,5 zakłada balans klas. Tu próg maksymalizujący F1 dobierany osobno dla każdej emocji na zbiorze val.

In [ ]:
def find_optimal_thresholds(y_true, y_proba, labels=None):
    return optimal_thresholds(y_true, y_proba)

In [ ]:
# --- Eksperyment 3: strojenie progów per-emocja na val (LogReg + tfidf_char z Eksp. 1) ---
clf_thr = OneVsRestClassifier(make_base(), n_jobs=1)
clf_thr.fit(X_train_b, y_tw_train)
y_proba_val = clf_thr.predict_proba(X_val_b)

default_metrics = evaluate(y_tw_val, (y_proba_val >= 0.5).astype(int))
opt_thresholds  = find_optimal_thresholds(y_tw_val, y_proba_val, EMOTIONS)
opt_metrics     = evaluate(y_tw_val, (y_proba_val >= opt_thresholds).astype(int))

thr_df = pd.DataFrame({
    "emotion":           EMOTIONS,
    "support_val":       y_tw_val.sum(axis=0),
    "optimal_threshold": np.round(opt_thresholds, 2),
    "f1_default_0.5":    [round(f1_score(y_tw_val[:, i], (y_proba_val[:, i] >= 0.5).astype(int),
                                         zero_division=0), 3) for i in range(len(EMOTIONS))],
    "f1_optimal":        [round(f1_score(y_tw_val[:, i], (y_proba_val[:, i] >= opt_thresholds[i]).astype(int),
                                         zero_division=0), 3) for i in range(len(EMOTIONS))],
})
thr_df["f1_lift"] = (thr_df["f1_optimal"] - thr_df["f1_default_0.5"]).round(3)
thr_df.to_csv(RESULTS_DIR / "exp3_thresholds.csv", index=False)

print(f"F1-Macro (val): default 0.5 = {default_metrics['f1_macro']:.3f} -> "
      f"optymalne progi = {opt_metrics['f1_macro']:.3f} "
      f"({opt_metrics['f1_macro'] - default_metrics['f1_macro']:+.3f})")
display(thr_df)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(EMOTIONS))
w = 0.35
ax.bar(x - w / 2, thr_df["f1_default_0.5"], w, label="Próg = 0,5", color="lightcoral")
ax.bar(x + w / 2, thr_df["f1_optimal"],     w, label="Próg optymalny", color="cornflowerblue")
ax.set_xticks(x)
ax.set_xticklabels(EMOTIONS, rotation=30, ha="right")
ax.set_ylabel("F1")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "exp3_thresholds.pdf", bbox_inches="tight")
plt.show()

In [15]:
# --- Handoff do 03c: metryki walidacyjne z eksperymentu 3 ---
json.dump(
    {"opt_metrics": {k: float(v) for k, v in opt_metrics.items()},
     "default_metrics": {k: float(v) for k, v in default_metrics.items()}},
    open(RESULTS_DIR / "exp3_handoff.json", "w"),
)
print("Saved exp3_handoff.json")

Saved exp3_handoff.json


In [16]:
# --- Log Eksp. 1-3 to MLflow ---
log_runs_from_df(tw_grid, "exp1_grid", ["representation", "model"],
                 tag_cols=["representation", "model"],
                 extra_tags={"dataset": "twitteremo", "split": "val"})
log_runs_from_df(ml_df, "exp2_multilabel", ["strategy"],
                 tag_cols=["strategy"], extra_tags={"dataset": "twitteremo", "split": "val"})

# Exp3: default vs optimal thresholds (metrics are dicts, not a DataFrame)
for nm, m in [("default", default_metrics), ("optimal", opt_metrics)]:
    with mlflow.start_run(run_name=f"exp3_threshold:{nm}"):
        mlflow.set_tag("phase", "exp3_threshold")
        mlflow.set_tag("threshold_strategy", nm)
        mlflow.set_tag("dataset", "twitteremo")
        mlflow.set_tag("split", "val")
        mlflow.log_metrics({k: float(v) for k, v in m.items()})

print("Logged Eksp. 1-3 to MLflow.  UI:  mlflow ui --backend-store-uri file:../mlruns")

Logged Eksp. 1-3 to MLflow.  UI:  mlflow ui --backend-store-uri file:../mlruns
